# 02 · Entrenamiento principal InterGATE V8

Ejecuta o reanuda las cinco semillas preespecificadas. Cada semilla guarda
su checkpoint, grafo predictivo, red biológica, auditoría de adaptación y
modelo de fase 2. Todas las métricas de este cuaderno proceden únicamente
de la partición de desarrollo.

Cambios respecto de V7:

- embedding entrenable de identidad de gen;
- supervisión profunda de expertos (`lambda=0.20`);
- rama tabular sobre el universo completo de genes conectados y anchura 256;
- dropout de la rama tabular (`p=0.30`);
- pérdida auxiliar directa de la GNN (`lambda=0.50`).

La ablación final mide si el clasificador utiliza cada rama. No demuestra
por sí sola que la topología aprendida sea superior a una topología nula.
`force=False` reanuda únicamente checkpoints cuyo fingerprint completo
coincide con V8; un paquete nuevo comienza necesariamente desde cero.


## 1 · Protocolo y rutas

In [ ]:
from pathlib import Path
import inspect, json, sys
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "intergate").is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Ejecute este notebook dentro de InterGATE_V8.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from intergate.config import (
    CFG, implementation_fingerprint, model_config_fingerprint,
)
from intergate.model import HybridGNNTabular, build_pruned_model_from
from intergate.runtime_setup import (
    assert_v8_protocol, assert_transport_protocol, prepare_runtime,
)
from intergate.training import finetune_pruned, train_one_epoch

ARTIFACT_NAME = "INTERGATE_V8"
RESULTS_NAME = "results_v8"

def assert_codigo_v8() -> None:
    pendientes = []
    for arg in (
        "use_node_identity", "nodes_used_full", "tab_in_dim",
        "tab_hidden_dim", "branch_dropout_tab", "branch_dropout_gnn",
    ):
        if arg not in inspect.signature(build_pruned_model_from).parameters:
            pendientes.append(f"model.py: build_pruned_model_from sin {arg}")
    for fn, arg in (
        (train_one_epoch, "expert_lambda"),
        (train_one_epoch, "gnn_aux_lambda"),
        (finetune_pruned, "expert_lambda"),
        (finetune_pruned, "gnn_aux_lambda"),
    ):
        if arg not in inspect.signature(fn).parameters:
            pendientes.append(f"training.py: {fn.__name__} sin {arg}")
    forward_source = inspect.getsource(HybridGNNTabular.forward)
    if "x_tab" not in forward_source:
        pendientes.append("model.py: forward no separa el bloque tabular")
    if "last_gnn_logits" not in forward_source:
        pendientes.append("model.py: forward no expone last_gnn_logits")
    if pendientes:
        raise RuntimeError("Módulos V8 incompletos:\n- " + "\n- ".join(pendientes))

assert_v8_protocol(CFG)
assert_transport_protocol(CFG)
assert_codigo_v8()

PHASE1_SAMPLES_PER_FLOW = (
    int(CFG.GRAPH_BATCH_SIZE) * int(CFG.TRANSPORT_STEPS_PER_EPISODE)
)
PHASE1_OPTIMIZER_STEPS = (
    int(CFG.TRANSPORT_STEPS_PER_EPISODE)
    + int(CFG.GRAPH_ACCUM_STEPS) - 1
) // int(CFG.GRAPH_ACCUM_STEPS)
SUPPORT_SAMPLES_PER_STEP = (
    int(CFG.GRAPH_BATCH_SIZE) * int(CFG.TRANSPORT_SUPPORT_BATCHES)
)
if (PHASE1_SAMPLES_PER_FLOW, PHASE1_OPTIMIZER_STEPS,
        SUPPORT_SAMPLES_PER_STEP) != (384, 2, 128):
    raise RuntimeError(
        "Presupuesto V8.1 incorrecto: "
        f"{PHASE1_SAMPLES_PER_FLOW} muestras/flujo, "
        f"{PHASE1_OPTIMIZER_STEPS} updates y "
        f"{SUPPORT_SAMPLES_PER_STEP} muestras de soporte."
    )

ARTIFACTS = Path(CFG.ARTIFACTS_ROOT)
RUN_ROOT = ARTIFACTS / ARTIFACT_NAME
RESULTS = PROJECT_ROOT / RESULTS_NAME
RESULTS.mkdir(parents=True, exist_ok=True)
print("Protocolo V8:", CFG.TRANSPORT_PROTOCOL_VERSION)
print("Fingerprint:", model_config_fingerprint(CFG))
print("Implementación:", implementation_fingerprint())
print("Artefactos:", RUN_ROOT)
print("Resultados:", RESULTS)


## 2 · Runtime de desarrollo, externo bloqueado

In [ ]:
from intergate.transport import TransportConfig, run_transport_seed

runtime = prepare_runtime(CFG, include_external=False)
if runtime["external_opened"] or len(runtime["test_idx"]) != 0:
    raise RuntimeError("El externo no esta bloqueado.")

transport = TransportConfig.from_global(CFG)
SEEDS = tuple(map(int, CFG.STABILITY_SEEDS))
RESUME = True
FORCE = False

CLASSES = [str(n) for n in runtime["classes"]]
LABEL_MAP = runtime["label_map"]

print("Semillas:", SEEDS)
print("Salida:", RUN_ROOT)
print(f"train={len(runtime['train_idx'])} val={len(runtime['val_idx'])} "
      f"test={len(runtime['test_idx'])}")
print(f"HIDDEN={CFG.HIDDEN} HYBRID_TAB_HIDDEN={CFG.HYBRID_TAB_HIDDEN} | "
      f"branch_drop_tab={CFG.BRANCH_DROPOUT_TAB} "
      f"gnn_aux={CFG.GNN_AUX_LOSS_LAMBDA} "
      f"expert={CFG.EXPERT_DEEP_SUPERVISION_LAMBDA}")
print(f"GRAPH_BATCH_SIZE={CFG.GRAPH_BATCH_SIZE} | "
      f"STEPS_PER_EPISODE={CFG.TRANSPORT_STEPS_PER_EPISODE} | "
      f"GRAPH_ACCUM_STEPS={CFG.GRAPH_ACCUM_STEPS}")
print(f"Contrato pareado: {PHASE1_SAMPLES_PER_FLOW} muestras por flujo, "
      f"{PHASE1_OPTIMIZER_STEPS} updates por episodio; "
      f"soporte={CFG.TRANSPORT_SUPPORT_BATCHES}x{CFG.GRAPH_BATCH_SIZE}="
      f"{SUPPORT_SAMPLES_PER_STEP}")


## 3 · Entrenamiento

En la traza `[FT-B]`, `GNN-solo macro_f1` es el diagnostico del equilibrio de
ramas: debe subir hacia 0.80-0.86 y acercarse al de la fusion. Si se queda
plano cerca de 0.25 en alguna semilla, esa semilla ha vuelto al colapso.

In [ ]:
rows = []
for position, seed in enumerate(SEEDS, 1):
    seed = int(seed)
    print(f"\n===== semilla {position}/{len(SEEDS)}: {seed} =====")
    row = run_transport_seed(
        seed,
        save_root=str(ARTIFACTS),
        transport=transport,
        resume=RESUME,
        force=FORCE,
        artifact_name=ARTIFACT_NAME,
    )
    rows.append(row)
    display(pd.DataFrame([row])[[
        "seed", "macro_f1", "bal_acc",
        "val_macro_f1_phase1_transport_consensus",
        "transport_best_validation_score_phase1",
        "transport_pairwise_jaccard_mean",
        "transport_median_rank_spearman",
        "transport_all_parameter_deltas_zero",
    ]])
    if bool(row["transport_all_parameter_deltas_zero"]):
        raise RuntimeError(
            f"Semilla {seed}: la adaptacion estructural es un no-op. "
            "No continue ni interprete Jaccard=1 como robustez."
        )

development_results = pd.DataFrame(rows).sort_values("seed")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
development_results.to_csv(RUN_ROOT / "development_results.csv", index=False)
display(development_results)

## 4 · Integridad y externo cerrado

In [ ]:
required = [
    "meta.json", "graph_bundle.pt", "biology_graph_bundle.pt",
    "model_pruned.pt", "transport_support_summary.json",
    "transport_adaptation_audit.csv",
]
missing = [
    str(RUN_ROOT / f"seed_{seed}" / name)
    for seed in SEEDS for name in required
    if not (RUN_ROOT / f"seed_{seed}" / name).is_file()
]
if missing:
    raise FileNotFoundError("Bundles incompletos:\n- " + "\n- ".join(missing))
if set(development_results.evaluation_split) != {"development_validation"}:
    raise RuntimeError("Una metrica no procede de desarrollo.")
if development_results.external_opened.astype(bool).any():
    raise RuntimeError("El externo fue abierto durante el entrenamiento.")

# Los flags no bastan: hay que ver las formas del state_dict, porque
# load_state_dict(strict=False) descarta en silencio lo que no encaja.
import torch

firmas = {}
for seed in SEEDS:
    m = torch.load(RUN_ROOT / f"seed_{int(seed)}" / "model_pruned.pt",
                   map_location="cpu", weights_only=False)
    f, sd = m["model_flags"], m["state_dict"]
    firmas[int(seed)] = {
        "node_identity": bool(f.get("use_node_identity", False)),
        "tab_in_dim": int(f.get("tab_in_dim", 0)),
        "tab_hidden": int(f.get("tab_hidden", 0)),
        "branch_drop_tab": float(f.get("branch_dropout_tab", 0.0)),
        "gnn_aux": float(f.get("gnn_aux_loss_lambda", 0.0)),
        "node_emb": tuple(sd["gnn.node_emb.weight"].shape),
        "tab_trunk.0": tuple(sd["tab_trunk.0.weight"].shape),
        "fusion.0": tuple(sd["fusion.0.weight"].shape),
    }
display(pd.DataFrame(firmas).T)

# node_emb tiene una fila por nodo de la union de los cuatro grafos Exact-K,
# y cada semilla aprende una topologia distinta: esa variacion es el objeto
# de estudio (la mide el Jaccard del cuaderno 03), no un fallo. Queda fuera
# de la comprobacion de homogeneidad.
COMPARABLES = ["node_identity", "tab_in_dim", "tab_hidden",
               "branch_drop_tab", "gnn_aux", "tab_trunk.0", "fusion.0"]

expected_tab_in = int(np.asarray(runtime["X_scaled"]).shape[1])
primera = firmas[int(SEEDS[0])]
if primera["tab_in_dim"] != expected_tab_in:
    raise RuntimeError("TAB_FULL_GENE_UNIVERSE no llego al modelo.")
if primera["tab_hidden"] != int(CFG.HYBRID_TAB_HIDDEN):
    raise RuntimeError("HYBRID_TAB_HIDDEN no llego al modelo.")
if primera["branch_drop_tab"] <= 0.0:
    raise RuntimeError("BRANCH_DROPOUT_TAB no llego al modelo.")
if primera["fusion.0"][1] != int(CFG.HIDDEN) + int(CFG.HYBRID_TAB_HIDDEN):
    raise RuntimeError(
        f"fusion.0 espera {int(CFG.HIDDEN) + int(CFG.HYBRID_TAB_HIDDEN)} "
        f"entradas y tiene {primera['fusion.0'][1]}."
    )

distintas = {
    tuple((k, firmas[int(s)][k]) for k in COMPARABLES) for s in SEEDS
}
if len(distintas) != 1:
    diferencias = {
        k: {int(s): firmas[int(s)][k] for s in SEEDS}
        for k in COMPARABLES
        if len({firmas[int(s)][k] for s in SEEDS}) > 1
    }
    raise RuntimeError(
        "Las semillas no comparten arquitectura: "
        + json.dumps(diferencias, ensure_ascii=False, default=str)
    )

tam = {int(s): firmas[int(s)]["node_emb"][0] for s in SEEDS}
print("Nodos del grafo por semilla (variacion esperada):", tam)
print(f"  min={min(tam.values())} max={max(tam.values())} "
      f"media={sum(tam.values()) / len(tam):.0f} de {expected_tab_in} genes "
      f"conectados ({100 * sum(tam.values()) / len(tam) / expected_tab_in:.1f} %)")
print("\nEntrenamiento principal V8 completo y externo cerrado.")


## 5 · Dependencia de ramas, identidad y mensajes en desarrollo

Se anula post hoc un componente cada vez sobre las mismas muestras. Además
de las ramas completas, se ponen a cero los pesos de message passing
conservando nodos y máscaras de clase, y se anula el embedding de identidad.
Esto separa mejor representación GNN, mensajes e identidad nodal.

Sigue siendo una perturbación del modelo ya entrenado: cuantifica
dependencia, pero no demuestra que la topología aprendida sea mejor que una
topología nula reentrenada. El dropout asimétrico también impide comparar las
magnitudes como si fueran importancias causales en una escala común.


In [ ]:
from contextlib import contextmanager, nullcontext
import torch

from intergate.ablation import build_pruned_model_from_bundle, load_bundle
from intergate.data import ExpressionDataset
from intergate.losses import compute_metrics_full
from intergate.training import predict_proba_xh_mode

X_FULL = np.asarray(runtime["X_scaled"], dtype=np.float32)
X_H = np.asarray(runtime["X_h"], dtype=np.float32)
Y = np.asarray(runtime["y"], dtype=np.int64)
VAL_IDX = np.asarray(runtime["val_idx"], dtype=np.int64)


@contextmanager
def mensajes_gnn_a_cero(model):
    """Anula mensajes conservando nodos, máscaras y parámetros aprendidos."""
    core = getattr(model, "gnn", model)
    names = ["edge_weight_norm_base"] + [
        f"fixed_class_edge_weight_norm_{class_id}"
        for class_id in range(int(getattr(core, "n_classes", 0)))
    ]
    tensors = [getattr(core, name) for name in names if hasattr(core, name)]
    if not tensors:
        raise RuntimeError("No se encontraron pesos fijos de message passing.")
    backups = [tensor.detach().clone() for tensor in tensors]
    with torch.no_grad():
        for tensor in tensors:
            tensor.zero_()
    try:
        yield
    finally:
        with torch.no_grad():
            for tensor, backup in zip(tensors, backups):
                tensor.copy_(backup)


@contextmanager
def identidad_nodal_a_cero(model):
    core = getattr(model, "gnn", model)
    embedding = getattr(core, "node_emb", None)
    if embedding is None:
        raise RuntimeError("V8 no expone el embedding de identidad nodal.")
    backup = embedding.weight.detach().clone()
    with torch.no_grad():
        embedding.weight.zero_()
    try:
        yield
    finally:
        with torch.no_grad():
            embedding.weight.copy_(backup)


def ablacion_validacion(seed):
    bundle = load_bundle(str(RUN_ROOT / f"seed_{int(seed)}"))
    model = build_pruned_model_from_bundle(bundle, device=CFG.DEVICE)
    model.eval()
    nodes = np.asarray(bundle["graph"]["nodes_used_full"], dtype=np.int64)
    flags = bundle.get("model", {}).get("model_flags", {})
    X_comp = (
        np.hstack([X_FULL[:, nodes], X_FULL]).astype(np.float32)
        if bool(flags.get("tab_full_gene_universe", False))
        else X_FULL[:, nodes]
    )
    loader = torch.utils.data.DataLoader(
        ExpressionDataset(X_comp, X_H, Y, VAL_IDX),
        batch_size=int(CFG.BATCH_SIZE), shuffle=False, num_workers=0,
    )
    variants = (
        ("completo", "orig", None),
        ("sin X_h", "zero", None),
        ("sin rama tabular", "orig", "tab"),
        ("sin GNN", "orig", "gnn"),
        ("sin message passing", "orig", "messages"),
        ("sin identidad nodal", "orig", "identity"),
    )
    rows = {}
    for name, xh_mode, disabled in variants:
        module = (
            model.tab_trunk if disabled == "tab"
            else model.gnn.trunk if disabled == "gnn"
            else None
        )
        hook = (
            module.register_forward_hook(
                lambda _module, _inputs, output: torch.zeros_like(output)
            )
            if module is not None else None
        )
        context = (
            mensajes_gnn_a_cero(model) if disabled == "messages"
            else identidad_nodal_a_cero(model) if disabled == "identity"
            else nullcontext()
        )
        try:
            with context, torch.no_grad():
                probability, truth = predict_proba_xh_mode(
                    model, None, loader, CFG.DEVICE, xh_mode=xh_mode,
                )
            rows[name] = compute_metrics_full(truth, probability, LABEL_MAP)
        finally:
            if hook is not None:
                hook.remove()
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return rows


all_rows = []
for seed in SEEDS:
    print(f"  ablación semilla {seed}...", end=" ", flush=True)
    for variant, metrics in ablacion_validacion(seed).items():
        all_rows.append({"seed": int(seed), "variante": variant, **metrics})
    print("ok")

ablacion = pd.DataFrame(all_rows)
ablacion.to_csv(RESULTS / "02_ablacion_ramas_validacion.csv", index=False)

order = [
    "completo", "sin X_h", "sin rama tabular", "sin GNN",
    "sin message passing", "sin identidad nodal",
]
metric_columns = ["acc", "bal_acc", "macro_f1", "auc_macro_ovr"]
display(
    ablacion.groupby("variante", sort=False)[metric_columns]
    .agg(["mean", "std"]).reindex(order).round(4)
)

base_f1 = (
    ablacion.loc[ablacion.variante.eq("completo")]
    .set_index("seed")["macro_f1"]
)
base_auc = (
    ablacion.loc[ablacion.variante.eq("completo")]
    .set_index("seed")["auc_macro_ovr"]
)
drops = []
for name in order[1:]:
    group = ablacion.loc[ablacion.variante.eq(name)].set_index("seed")
    delta_f1 = base_f1 - group["macro_f1"]
    delta_auc = base_auc - group["auc_macro_ovr"]
    drops.append({
        "bloque_apagado": name,
        "caida_macro_f1": float(delta_f1.mean()),
        "sd_macro_f1": float(delta_f1.std()),
        "semillas_caida_macro_f1_positiva": int((delta_f1 > 0).sum()),
        "caida_auc": float(delta_auc.mean()),
        "sd_auc": float(delta_auc.std()),
        "semillas_caida_auc_positiva": int((delta_auc > 0).sum()),
        "semillas": int(len(group)),
    })
caida_tabla = pd.DataFrame(drops)
caida_tabla.to_csv(
    RESULTS / "02_caida_por_bloque_validacion.csv", index=False,
)
display(caida_tabla.round(4))

indexed = caida_tabla.set_index("bloque_apagado")
c_gnn = float(indexed.loc["sin GNN", "caida_auc"])
c_tab = float(indexed.loc["sin rama tabular", "caida_auc"])
c_messages = float(indexed.loc["sin message passing", "caida_auc"])
print(
    f"\nCaída de AUROC: GNN {c_gnn:.4f} | mensajes {c_messages:.4f} "
    f"| rama tabular {c_tab:.4f}"
)
if c_gnn > 0:
    print("La representación GNN participa en la predicción de validación.")
else:
    print("AVISO: no se observa dependencia positiva de la rama GNN.")
if c_messages > 0:
    print("Los mensajes sobre las aristas contribuyen dentro del modelo entrenado.")
else:
    print("AVISO: no se observa una contribución positiva de los mensajes.")
print(
    "Son perturbaciones post hoc: no prueban que la topología aprendida "
    "supere a una topología nula reentrenada."
)


## 6 · Manifiesto

In [ ]:
manifest = {
    "status": "complete",
    "stage": "02_main_training",
    "artifact_name": ARTIFACT_NAME,
    "model_version": "InterGATE-V8.1",
    "protocol_version": str(CFG.TRANSPORT_PROTOCOL_VERSION),
    "model_config_sha256": model_config_fingerprint(CFG),
    "implementation_sha256": implementation_fingerprint(),
    "external_opened_during_training": False,
    "evaluation_split": "development_validation",
    "seeds": list(map(int, SEEDS)),
    "n_train": int(len(runtime["train_idx"])),
    "n_val": int(len(runtime["val_idx"])),
    "config": {
        "HIDDEN": int(CFG.HIDDEN),
        "HYBRID_TAB_HIDDEN": int(CFG.HYBRID_TAB_HIDDEN),
        "USE_NODE_IDENTITY": bool(CFG.USE_NODE_IDENTITY),
        "TAB_FULL_GENE_UNIVERSE": bool(CFG.TAB_FULL_GENE_UNIVERSE),
        "EXPERT_DEEP_SUPERVISION_LAMBDA": float(
            CFG.EXPERT_DEEP_SUPERVISION_LAMBDA),
        "BRANCH_DROPOUT_TAB": float(CFG.BRANCH_DROPOUT_TAB),
        "BRANCH_DROPOUT_GNN": float(CFG.BRANCH_DROPOUT_GNN),
        "GNN_AUX_LOSS_LAMBDA": float(CFG.GNN_AUX_LOSS_LAMBDA),
        "KEEP_MIN": float(CFG.KEEP_MIN),
        "GRAPH_BATCH_SIZE": int(CFG.GRAPH_BATCH_SIZE),
        "GRAPH_ACCUM_STEPS": int(CFG.GRAPH_ACCUM_STEPS),
        "TRANSPORT_STEPS_PER_EPISODE": int(
            CFG.TRANSPORT_STEPS_PER_EPISODE),
        "PHASE1_SAMPLES_PER_FLOW": int(PHASE1_SAMPLES_PER_FLOW),
        "PHASE1_OPTIMIZER_STEPS": int(PHASE1_OPTIMIZER_STEPS),
        "TRANSPORT_SUPPORT_BATCHES": int(
            CFG.TRANSPORT_SUPPORT_BATCHES),
        "SUPPORT_SAMPLES_PER_STEP": int(SUPPORT_SAMPLES_PER_STEP),
    },
    "branch_ablation_validation": caida_tabla.to_dict(orient="records"),
    "branch_ablation_interpretation": (
        "posthoc branch reliance; not causal importance or topology proof"
    ),
    "component_ablation_interpretation": (
        "GNN, tabular, X_h, message weights and node identity are "
        "ablated separately on development validation"
    ),
}
(RESULTS / "02_training_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(json.dumps(manifest["config"], indent=2, ensure_ascii=False))
print("\nManifiesto en:", RESULTS / "02_training_manifest.json")
